In [1]:
from IPython.display import display, Markdown, HTML
import circuitsvis as cv
import torch as t
import pandas as pd
from itables import init_notebook_mode, show as itshow
init_notebook_mode(all_interactive=True)
from shared import (
    load_model, run_and_cache, get_attention_pattern,
    show_head_pattern, show_attention_tables, show_attention_to_position,
    show_self_attention_pcts, show_prev_token_pcts,
    show_attention_to_token, show_few_prev_tokens_pcts,
    compute_head_raw_pcts, attention_to_token_pcts, few_prev_tokens_pcts,
    HEAD_CLASSIFICATIONS, HEAD_TYPES, TYPE_TO_HEADS,
    ACTIVITY_LABELS, ACTIVITY_ORDER,
    get_head_types, TEXT,
)

/home/burny/projects/ai/mechinterp/attention-head-zoo-2-layer-attention-only-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Introducing Our Toy Attention-Only Model

Here we introduce a toy 2L attention-only transformer. Some changes to make them easier to interpret:
- It has only attention blocks.
- The positional embeddings are only added to the residual stream before calculating each key and query vector in the attention layers as opposed to the token embeddings - i.e. we compute queries as `Q = (resid + pos_embed) @ W_Q + b_Q` and same for keys, but values as `V = resid @ W_V + b_V`. This means that **the residual stream can't directly encode positional information**.
    - This turns out to make it *way* easier for induction heads to form, it happens 2-3x times earlier - [see the comparison of two training runs](https://wandb.ai/mechanistic-interpretability/attn-only/reports/loss_ewma-22-08-24-11-08-83---VmlldzoyNTI0MDMz?accessToken=8ap8ir6y072uqa4f9uinotdtrwmoa8d8k2je4ec0lyasf1jcm3mtdh37ouijgdbm) here. (The bump in each curve is the formation of induction heads.)
    - The argument that does this below is `positional_embedding_type="shortformer"`.
- It has no MLP layers, no LayerNorms, and no biases.
- There are separate embed and unembed matrices (i.e. the weights are not tied).

We now define our model with a `HookedTransformerConfig` object. 

In [2]:
model = load_model()

In [3]:
str_tokens, logits, cache = run_and_cache(model)

## All Attention Patterns by Layer

In [4]:
attention_patterns_for_all_layers = t.stack([
    cache["pattern", layer] for layer in range(model.cfg.n_layers)
])

for layer in range(model.cfg.n_layers):
    attention_pattern = attention_patterns_for_all_layers[layer]
    print(f"Layer {layer} Head Attention Patterns:")
    display(
        cv.attention.attention_patterns(
            tokens=str_tokens,
            attention=attention_pattern,
        )
    )
    display(
        cv.attention.attention_heads(
            tokens=str_tokens,
            attention=attention_pattern,
            attention_head_names=[f"L{layer}H{i}" for i in range(12)],
        )
    )

Layer 0 Head Attention Patterns:


Layer 1 Head Attention Patterns:


## Per-Head Classifications


**Layer 0:**

- **L0H0**: Attends to glue words like "are", "and", "if", "that", "were", "or"
- **L0H1**: Attends from and to words representing certainty and questioning (if, likely, are, think, known to, how) and some stuff from . consistently
- **L0H2**: Cares primarily about end of text token and some ,
- **L0H3**: Cares only about end of text token
- **L0H4**: Attends to few previous tokens
- **L0H5**: Attends to just stuff like ,
- **L0H6**: Attends to only end of text token
- **L0H7**: Attends only to previous token, one to semantically salient (scaled up)
- **L0H8**: Attends to glue words like "are", "and", "if", "that", "were", "or"
- **L0H9**: Attends to itself and to previous token and to few previous tokens
- **L0H10**: Glue words, certainty, end of text
- **L0H11**: To itself, glue words, end of text, semantically salient (scaled up, deceptive), certainty

**Layer 1:**

- **L1H0**: End of text token, itself, token ","
- **L1H1**: End of text, previous token, glue words
- **L1H2**: End of text, self-attention, content words attend to EOT
- **L1H3**: Primarily end of text token, diffuse across positions
- **L1H4**: Cares only about end of text token
- **L1H5**: Aggregating all context into rich words? and dot somehow cares about end of text wtf
- **L1H6**: Connector between glue words and semantically rich words :D and some other types already listed
- **L1H7**: Connector between glue words and semantically rich words (and other glue words?) (1x partial)
- **L1H8**: Attending to previous token if directly related and other types already listed
- **L1H9**: Diffuse attention across context, some end of text, broad aggregation
- **L1H10**: Connector between glue words and semantically rich words, semantic connector between two related things (machine intelligence)
- **L1H11**: End of text for function words, glue words, previous token

## Type Summary

- **End-of-Text Attender** (11x): L1H4 (full), L0H3 (full), L0H6 (fullish), L1H3 (half), L1H2 (partial), L1H11 (partial), L0H2 (partial), L0H10 (partial), L0H11 (partial), L1H0 (partial), L1H9 (partial)
- **Glue Word Attender** (6x): L0H0 (full), L0H8 (full), L0H10 (partial), L0H11 (partial), L1H1 (partial), L1H11 (partial)
- **Previous Token Head** (4x): L0H7 (full), L0H9 (partial), L1H1 (partial), L1H11 (partial)
- **Certainty/Questioning Attender** (3x): L0H1 (half), L0H10 (partial), L0H11 (partial)
- **Self-Attender** (3x): L1H2 (partial), L0H9 (partial), L0H11 (partial)
- **Glue-to-Semantic Connector** (3x): L1H6 (half), L1H7 (half), L1H10 (half)
- **Few Previous Tokens Head** (2x): L0H4 (full), L0H9 (partial)
- **Comma Attender** (2x): L0H5 (full), L0H2 (partial)
- **Semantically Salient Attender** (2x): L0H7 (half), L0H11 (partial)
- **Context Aggregator** (2x): L1H5 (half), L1H9 (partial)
- **Period Attender** (1x): L1H5 (half)
- **Dot-EOT Quirk** (1x): L1H5 (half)
- **Glue-to-Glue Connector** (1x): L1H7 (partial)
- **Related Previous Token** (1x): L1H8 (partial)
- **Semantic Connector** (1x): L1H10 (partial)

## Activity Levels

- full (100-90%)
- fullish (90-60%)
- half (60-40%)
- partial (40-10%)
- almost none (10-0.1%)

## Summary: All 24 Attention Heads

Programmatic summary pulling classifications and type mappings from `shared.py`. See `heads/` for per-head notebooks and `types/` for per-type notebooks.

### Per-Head Classification Table

In [5]:
# Per-head summary: classification + types with activity levels + raw %
raw_pcts = compute_head_raw_pcts(cache)
comma_pcts = {(l, h): pct for l, h, pct, _ in attention_to_token_pcts(cache, str_tokens, ",")}
period_pcts = {(l, h): pct for l, h, pct, _ in attention_to_token_pcts(cache, str_tokens, ".")}
few_prev_pcts = {(l, h): pct for l, h, pct, _ in few_prev_tokens_pcts(cache, k=5)}

rows = []
for layer in range(2):
    for head in range(12):
        head_types = get_head_types(layer, head)
        types_str = ", ".join(
            f"{HEAD_TYPES[tid][0]} ({ACTIVITY_LABELS[act]})"
            for tid, act in head_types
        ) if head_types else "\u2014"
        pcts = raw_pcts[(layer, head)]
        rows.append({
            "Head": f"L{layer}H{head}",
            "Classification": HEAD_CLASSIFICATIONS.get((layer, head), "\u2014"),
            "Types": types_str,
            "EOT %": round(pcts["eot"], 1),
            "Self %": round(pcts["self_attn"], 1),
            "Prev %": round(pcts["prev_token"], 1),
            "Comma %": round(comma_pcts[(layer, head)], 1),
            "Period %": round(period_pcts[(layer, head)], 1),
            "Prev5 %": round(few_prev_pcts[(layer, head)], 1),
        })
df = pd.DataFrame(rows)
itshow(df, paging=False, classes="display compact")

Loading ITables v2.7.0 from the init_notebook_mode cell... (need help?)


### Per-Type Summary

Each type with the number of heads exhibiting it, sorted by head count descending.

In [6]:
# Per-type summary: sorted by number of heads descending
# Compute all measurable metrics
raw_pcts = compute_head_raw_pcts(cache)
comma_pcts = {(l, h): pct for l, h, pct, _ in attention_to_token_pcts(cache, str_tokens, ",")}
period_pcts = {(l, h): pct for l, h, pct, _ in attention_to_token_pcts(cache, str_tokens, ".")}
few_prev_pcts = {(l, h): pct for l, h, pct, _ in few_prev_tokens_pcts(cache, k=5)}

def get_metric(type_id, l, h):
    if type_id == "end_of_text": return raw_pcts[(l, h)]["eot"]
    if type_id == "self_attention": return raw_pcts[(l, h)]["self_attn"]
    if type_id == "previous_token": return raw_pcts[(l, h)]["prev_token"]
    if type_id == "comma_attention": return comma_pcts[(l, h)]
    if type_id == "period_attention": return period_pcts[(l, h)]
    if type_id == "few_previous_tokens": return few_prev_pcts[(l, h)]
    return None

rows = []
for type_id, heads_list in TYPE_TO_HEADS.items():
    display_name, description = HEAD_TYPES[type_id]
    has_metric = get_metric(type_id, heads_list[0][0][0], heads_list[0][0][1]) is not None
    if has_metric:
        sorted_heads = sorted(
            heads_list,
            key=lambda x: get_metric(type_id, x[0][0], x[0][1]),
            reverse=True,
        )
        heads_str = ", ".join(
            f"L{l}H{h} ({get_metric(type_id, l, h):.1f}%)"
            for (l, h), act in sorted_heads
        )
    else:
        sorted_heads = sorted(heads_list, key=lambda x: ACTIVITY_ORDER.get(x[1], 0), reverse=True)
        heads_str = ", ".join(f"L{l}H{h} ({act})" for (l, h), act in sorted_heads)
    rows.append({
        "Type": display_name,
        "Description": description,
        "# Heads": len(heads_list),
        "Heads (by activity)": heads_str,
    })
df = pd.DataFrame(rows).sort_values("# Heads", ascending=False).reset_index(drop=True)
itshow(df, paging=False, classes="display compact")

Loading ITables v2.7.0 from the init_notebook_mode cell... (need help?)


### Head-Type Matrix

Which heads exhibit which types. Values show raw attention % where computable, ~midpoint estimates for non-measurable types.

In [7]:
import plotly.graph_objects as go

# Compute raw % for all 6 measurable types
raw_pcts = compute_head_raw_pcts(cache)
comma_pcts = {(l, h): pct for l, h, pct, _ in attention_to_token_pcts(cache, str_tokens, ",")}
period_pcts = {(l, h): pct for l, h, pct, _ in attention_to_token_pcts(cache, str_tokens, ".")}
few_prev_pcts = {(l, h): pct for l, h, pct, _ in few_prev_tokens_pcts(cache, k=5)}

def get_raw_pct(tid, l, h):
    if tid == "end_of_text": return raw_pcts[(l, h)]["eot"]
    if tid == "self_attention": return raw_pcts[(l, h)]["self_attn"]
    if tid == "previous_token": return raw_pcts[(l, h)]["prev_token"]
    if tid == "comma_attention": return comma_pcts[(l, h)]
    if tid == "period_attention": return period_pcts[(l, h)]
    if tid == "few_previous_tokens": return few_prev_pcts[(l, h)]
    return None

# Map activity levels to midpoint % for non-measurable types
ACTIVITY_MIDPOINT = {"full": 95, "fullish": 75, "half": 50, "partial": 25, "almost_none": 5}

type_ids = sorted(HEAD_TYPES.keys(), key=lambda tid: len(TYPE_TO_HEADS.get(tid, [])), reverse=True)
head_names = [f"L{l}H{h}" for l in range(2) for h in range(12)]

z = []
text_matrix = []
for tid in type_ids:
    row = []
    text_row = []
    heads_map = {(l, h): act for (l, h), act in TYPE_TO_HEADS.get(tid, [])}
    for l in range(2):
        for h in range(12):
            if (l, h) in heads_map:
                raw = get_raw_pct(tid, l, h)
                if raw is not None:
                    row.append(raw)
                    text_row.append(f"{raw:.0f}")
                else:
                    val = ACTIVITY_MIDPOINT[heads_map[(l, h)]]
                    row.append(val)
                    text_row.append(f"~{val}")
            else:
                row.append(0)
                text_row.append("")
    z.append(row)
    text_matrix.append(text_row)

type_labels = [HEAD_TYPES[tid][0] for tid in type_ids]

fig = go.Figure(data=go.Heatmap(
    z=z,
    x=head_names,
    y=type_labels,
    colorscale=[
        [0, "#f8f8f8"],
        [0.01, "#fff5f0"],
        [0.1, "#fde0dd"],
        [0.25, "#fa9fb5"],
        [0.4, "#f768a1"],
        [0.6, "#c51b8a"],
        [0.8, "#7a0177"],
        [1.0, "#49006a"],
    ],
    zmin=0,
    zmax=100,
    text=text_matrix,
    texttemplate="%{text}",
    hovertemplate="Head: %{x}<br>Type: %{y}<br>Activity: %{z:.1f}%<extra></extra>",
))

fig.update_layout(
    title="Head-Type Activity Matrix (% attention weight)",
    xaxis_title="Attention Head",
    yaxis_title="Type",
    height=600,
    width=1000,
    yaxis=dict(autorange="reversed"),
)
fig.show()

### Layer-Level Observations

Summary of functional differences between Layer 0 and Layer 1 heads.

In [8]:
# Count type assignments per layer
from collections import Counter

for layer in range(2):
    print(f"\n=== Layer {layer} ===")
    type_counts = Counter()
    for type_id, heads_list in TYPE_TO_HEADS.items():
        for (l, h), activity in heads_list:
            if l == layer:
                type_counts[HEAD_TYPES[type_id][0]] += 1
    for type_name, count in type_counts.most_common():
        print(f"  {type_name}: {count} head(s)")

print("\n=== Layer comparison ===")
layer0_types = set()
layer1_types = set()
for type_id, heads_list in TYPE_TO_HEADS.items():
    for (l, h), activity in heads_list:
        if l == 0:
            layer0_types.add(HEAD_TYPES[type_id][0])
        else:
            layer1_types.add(HEAD_TYPES[type_id][0])

only_l0 = layer0_types - layer1_types
only_l1 = layer1_types - layer0_types
print(f"Types only in Layer 0: {only_l0 or 'none'}")
print(f"Types only in Layer 1: {only_l1 or 'none'}")
print(f"Types in both layers: {layer0_types & layer1_types}")


=== Layer 0 ===
  End-of-Text Attender: 5 head(s)
  Glue Word Attender: 4 head(s)
  Certainty/Questioning Attender: 3 head(s)
  Few Previous Tokens Head: 2 head(s)
  Previous Token Head: 2 head(s)
  Comma Attender: 2 head(s)
  Self-Attender: 2 head(s)
  Semantically Salient Attender: 2 head(s)

=== Layer 1 ===
  End-of-Text Attender: 6 head(s)
  Glue-to-Semantic Connector: 3 head(s)
  Glue Word Attender: 2 head(s)
  Previous Token Head: 2 head(s)
  Context Aggregator: 2 head(s)
  Period Attender: 1 head(s)
  Self-Attender: 1 head(s)
  Dot-EOT Quirk: 1 head(s)
  Glue-to-Glue Connector: 1 head(s)
  Related Previous Token: 1 head(s)
  Semantic Connector: 1 head(s)

=== Layer comparison ===
Types only in Layer 0: {'Semantically Salient Attender', 'Certainty/Questioning Attender', 'Comma Attender', 'Few Previous Tokens Head'}
Types only in Layer 1: {'Dot-EOT Quirk', 'Glue-to-Glue Connector', 'Context Aggregator', 'Glue-to-Semantic Connector', 'Period Attender', 'Semantic Connector', 'Relat